# Gold_price_distribution

## Import and Load

In [15]:
import pandas as pd

from ecommerce_ingestion.config.paths import DB_OUTPUT_GOLD
from ecommerce_ingestion.utils.genre_unifier_service import get_genre_unified_genre_data


In [16]:
gold_price_distribution_raw = pd.read_parquet(
    DB_OUTPUT_GOLD / "gold_game_catalog.parquet")

gold_price_distribution_raw.shape

(2976, 18)

In [17]:
gold_price_distribution_filtered = get_genre_unified_genre_data(
    gold_price_distribution_raw)
gold_price_distribution_filtered.shape

(4661, 19)

In [18]:
gold_price_distribution_filtered.head()

,game_id,game_name,game_developer,game_rating,game_product_type,genre_id,current_price,currency,stock_status,meta_score,user_score,category_name,category_parent_id,primary_genre,secondary_genre,tertiary_genre,price_bucket,score_bucket,unified_genre
0,the legend of zelda: ocarina of time,The Legend of Zelda: Ocarina of Time,nintendo,E,singleplayer,"action adventure, fantasy",91.99,EUR,in_stock,99,91,nintendo-64,nintendo,action,unknown,unknown,60+,masterpiece,action
1,super mario galaxy,Super Mario Galaxy,nintendo,E,singleplayer,"action, platformer, 3d",91.99,EUR,out_of_stock,97,91,wii,nintendo,action,platformer,unknown,60+,masterpiece,action
2,super mario galaxy 2,Super Mario Galaxy 2,nintendo ead tokyo,E,singleplayer,"action, platformer, 3d",91.99,EUR,in_stock,97,91,wii,nintendo,action,platformer,unknown,60+,masterpiece,action
3,metroid prime,Metroid Prime,retro studios,T,singleplayer,"action, shooter, first-person, sci-fi",89.99,EUR,out_of_stock,97,89,gamecube,nintendo,action,shooter,unknown,60+,masterpiece,action
4,super mario odyssey,Super Mario Odyssey,nintendo,E10+,singleplayer,"action, platformer, 3d",89.99,EUR,in_stock,97,89,switch,nintendo,action,platformer,unknown,60+,masterpiece,action


## Grouping

Columns

número de juegos
precio medio
meta score medio
user score medio
porcentaje en stock
plataformas donde aparece
índice de atractivo del género

In [19]:
gold_price_distribution = gold_price_distribution_filtered.groupby(["price_bucket", 
                                                           "category_name", 
                                                           "unified_genre"]).agg(
     game_count=("game_id", "count"),
     average_user_score=("user_score", "mean"),
     average_meta_score=("meta_score", "mean"),
     average_price=("current_price", "mean"),
     in_stock_percentage=("stock_status", lambda x: (x == "in_stock").mean() * 100)
 ).reset_index()
gold_price_distribution["score_price"] = gold_price_distribution[
    "average_user_score"] / gold_price_distribution["average_price"]
gold_price_distribution.shape

(411, 9)

In [20]:
gold_price_distribution.head()


,price_bucket,category_name,unified_genre,game_count,average_user_score,average_meta_score,average_price,in_stock_percentage,score_price
0,10-20,playstation-4,action,1,18.0,76.0,18.99,100.0,0.947867
1,10-20,playstation-4,shooter,1,18.0,76.0,18.99,100.0,0.947867
2,10-20,playstation-4,strategy,1,18.0,76.0,18.99,100.0,0.947867
3,10-20,xbox-one,adventure,1,19.0,77.0,19.99,0.0,0.950475
4,10-20,xbox-one,simulation,2,13.0,78.0,13.99,0.0,0.929235


In [21]:
gold_price_distribution.isna().sum()

price_bucket           0
category_name          0
unified_genre          0
game_count             0
average_user_score     0
average_meta_score     0
average_price          0
in_stock_percentage    0
score_price            0
dtype: int64

## Save Data

In [22]:
gold_price_distribution.to_parquet(DB_OUTPUT_GOLD / "gold_price_distribution.parquet")